# Runs de tâche : découpage et contrôle ERD

Question du projet : **bouger ou imaginer bouger, est-ce le même réseau cérébral ?** Avant de comparer des
graphes, on vérifie que la chaîne retrouve l'effet physiologique le mieux établi de ces runs.

| Condition | Runs EEGMMIDB | Epochs |
|---|---|---|
| MM : mouvement réel du poing gauche / droit | R03, R07, R11 | T1 (gauche), T2 (droite) |
| MI : imagination du même mouvement | R04, R08, R12 | T1 (gauche), T2 (droite) |
| REST : pauses entre essais | les 6 runs | T0 |

Découpage `TASK_LR` : fenêtre de 0.5 à 4.0 s après chaque annotation (T1/T2 durent 4.1 s, T0 4.2 s).

**Contrôle ERD** (Pfurtscheller & Lopes da Silva 1999) : pendant le mouvement réel et imaginé, la puissance
alpha/mu et bêta baisse sur le cortex sensorimoteur. Par sujet, condition et bande :
ERD(région) = moyenne des log10 P des epochs tâche (left + right) − moyenne des epochs rest.
ROI = precentral, postcentral, paracentral, des deux côtés.

- (a) la ROI est-elle dans les 10 régions les plus négatives (moyenne des sujets) ? Surreprésentation testée
  par loi hypergéométrique (6 régions ROI sur 68, 10 tirées).
- (b) Wilcoxon unilatéral sur les sujets : ERD moyenne de la ROI < 0, en MM et en MI.
- (c) Wilcoxon apparié : ERD de la ROI plus négative en MM qu'en MI.


In [ ]:
%matplotlib inline
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from eeg_graph import validation
from eeg_graph.config import TASK_LR, Paths, setup_logging
from eeg_graph.loading import load_eegmmidb_run
from eeg_graph.pipeline import DATASET, build_cache
from eeg_graph.preprocessing import epoch_onsets
from eeg_graph.viz import plot_region_values, save_figure

setup_logging()
paths = Paths()
paths.figures.mkdir(parents=True, exist_ok=True)
TABLES = paths.root / "reports" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

In [ ]:
SUBJECTS = list(range(1, 11))
RUNS = [3, 4, 7, 8, 11, 12]  # MM : 3, 7, 11 ; MI : 4, 8, 12
NODE_SPACE = "source"
BANDS = ("alpha", "beta")
TOP = 10  # contrôle (a)
N_JOBS = 4  # ~1.2 Go par worker (pic mesuré sur S001), 16 Go de RAM
OVERWRITE = False

## 1. Cache des runs de tâche

In [ ]:
summary = build_cache(SUBJECTS, RUNS, NODE_SPACE, n_jobs=N_JOBS, overwrite=OVERWRITE)
index = pd.read_csv(paths.cache / DATASET / "index.csv")
task_index = index[(index.epoch_spec == TASK_LR.name) & index.subject.isin(SUBJECTS) & index.run.isin(RUNS)]
display(task_index.groupby("status").size().rename("n"))
# temps par sujet : somme des temps par enregistrement de index.csv (préparation partagée comprise)
timing = task_index[task_index.status == "ok"].groupby("subject").seconds.sum().rename("seconds")
timing.describe().round(1)

**Epochs retenues par label et par run.** Les totaux viennent des annotations ; les retenues, du cache
(après rejet crête-à-crête). Un rejet très différent entre labels biaiserait l'ERD.

In [ ]:
task_results = validation.load_task_results(paths, SUBJECTS, NODE_SPACE, RUNS)
kept = validation.epoch_label_counts(task_results)
totals = []
for s, run in kept[["subject", "run"]].itertuples(index=False):
    _, labels, _ = epoch_onsets(load_eegmmidb_run(s, run, paths), TASK_LR)
    totals.append(
        {"subject": s, "run": run} | {f"{lab}_total": int((labels == lab).sum()) for lab in ("rest", "left", "right")}
    )
counts = kept.merge(pd.DataFrame(totals), on=["subject", "run"])
counts.to_csv(TABLES / "task_epoch_counts.csv", index=False)
by_run = counts.groupby("run")[["rest", "rest_total", "left", "left_total", "right", "right_total"]].sum()
by_run["kept_pct"] = (
    100
    * by_run[["rest", "left", "right"]].sum(axis=1)
    / by_run[["rest_total", "left_total", "right_total"]].sum(axis=1)
).round(1)
by_run

**Taille du cache** et extrapolation à 50 sujets × 6 runs.

In [ ]:
files = [paths.root / p for p in task_index[task_index.status == "ok"].path]
sizes_mb = np.array([f.stat().st_size for f in files]) / 1e6
cache_info = {
    "n_files": len(files),
    "median_file_mb": float(np.median(sizes_mb)),
    "max_file_mb": float(sizes_mb.max()),
    "total_mb": float(sizes_mb.sum()),
    "extrapolated_50x6_gb": float(np.median(sizes_mb) * 50 * 6 / 1e3),
    "format": "np.savez_compressed",
}
cache_info

## 2. ERD par région

In [ ]:
erds = validation.erd_by_subject(task_results, BANDS)
atlas = next(iter(next(iter(task_results.values())).values()))[0].atlas()
report = validation.task_erd_report(erds, atlas, top=TOP)
subjects_used = sorted(s for s, c in task_results.items() if all(k in c for k in validation.TASK_CONDITIONS))
rows = []
for cond, band in erds:
    a, b = report[f"{cond}_{band}"]["a_roi_in_top"], report[f"{cond}_{band}"]["b_roi_erd_negative"]
    rows.append(
        {
            "condition": cond,
            "band": band,
            "n_roi_in_top10": a["n_roi_in_top"],
            "hypergeom_p": a["hypergeom_p"],
            "roi_mean_erd": b["mean_erd_log10"],
            "power_change_pct": b["mean_power_change_pct"],
            "n_negative": b["n_subjects_negative"],
            "wilcoxon_p_less": b["wilcoxon_p_less"],
        }
    )
display(pd.DataFrame(rows).round(4))
pd.DataFrame({band: report[f"c_mm_vs_mi_{band}"] for band in BANDS}).T

In [ ]:
for (cond, band), values in erds.items():
    fig, ax = plt.subplots(figsize=(6, 5.5))
    plot_region_values(
        ax,
        values.mean(axis=0),
        atlas,
        f"ERD {band} {cond} : log10 P tâche − rest, moyenne de {len(values)} sujets",
        top=20,
        largest=False,
        highlight=validation.MOTOR_ROI,
    )
    ax.set_xlabel("log10 P tâche − log10 P rest (ROI en gras)")
    save_figure(fig, paths.figures / f"validation_task_erd_{cond}_{band}.png")
    plt.show()

In [ ]:
per_subject = pd.DataFrame(
    {f"{c}_{b}": validation.roi_mean(v, atlas) for (c, b), v in erds.items()}, index=subjects_used
)
per_subject.index.name = "subject"
per_subject.round(4).to_csv(TABLES / "task_erd_roi_by_subject.csv")
out = {
    "subjects": subjects_used,
    "runs": RUNS,
    "epoch_spec": {"name": TASK_LR.name, "tmin": TASK_LR.tmin, "tmax": TASK_LR.tmax, "events": dict(TASK_LR.events)},
    "epochs_by_run": by_run.reset_index().to_dict(orient="records"),
    "seconds_per_subject": {"median": float(timing.median()), "min": float(timing.min()), "max": float(timing.max())},
    "cache": cache_info,
    "erd": report,
    "roi_erd_by_subject": per_subject.round(5).reset_index().to_dict(orient="records"),
}
(paths.root / "reports" / "validation_task.json").write_text(json.dumps(out, indent=2, ensure_ascii=False))
pd.Series(
    {f"(a) {k}": v["a_roi_in_top"]["passed"] for k, v in report.items() if isinstance(v, dict) and "a_roi_in_top" in v}
    | {
        f"(b) {k}": v["b_roi_erd_negative"]["passed"]
        for k, v in report.items()
        if isinstance(v, dict) and "b_roi_erd_negative" in v
    },
    name="validé",
)